# Estimativa de Peso de Bovinos por Medidas Morfométricas

**Objetivo:** Comparar modelos de machine learning para prever o peso vivo de vacas Hereford a partir de medidas corporais simples, sem necessidade de balança industrial.

**Dataset:** Hereford_cows - Ruchay et al. (2022) | 1.523 vacas | 11 medidas corporais + idade

**Por que isso é relevante para a região da Campanha Gaucha?**  
A Campanha Gaúcha tem uma das maiores concentrações de pecuária de corte do país, com Hereford e Braford dominando os campos nativos do Pampa. Monitorar ganho de peso é parte do dia a dia do produtor, mas balança de precisão ainda é realidade de poucos. Com medidas tiradas no manejo rotineiro e um modelo treinado, qualquer propriedade consegue estimar o peso dos animais sem custo extra.

## 0. Instalação de Dependências

In [ ]:
!pip install shap xgboost --quiet
print('Dependências prontas')

## 1. Imports e Configuração

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from scipy import stats
from xgboost import XGBRegressor
import shap
import pickle, os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
SEED = 42
np.random.seed(SEED)
print('Imports OK')

## 2. Carregamento e Padronização

In [ ]:
df = pd.read_csv(
    'https://raw.githubusercontent.com/jota6k/projeto_peso_bovinos/main/data/Hereford_cows.csv',
    encoding='latin-1',
    sep=';'
)

df = df.drop(columns=['n.', 'identificator'])

df = df.rename(columns={
    'Age years':              'idade_anos',
    'Body weight':            'peso_kg',
    'withers height':         'altura_cernelha',
    'hip height':             'altura_garupa',
    'chest depth':            'profundidade_peito',
    'chest width':            'largura_peito',
    'heart girth':            'perimetro_toracico',
    'ilium width':            'largura_iliaca',
    'sciatic tubercle width': 'largura_isquiatica',
    'oblique body length':    'comp_obliquo_corpo',
    'oblique rear length':    'comp_obliquo_traseiro',
    'metacarpus girth':       'perimetro_metacarpo',
    'backside half-girth':    'meia_circ_traseira',
})

FEATURES = [
    'altura_cernelha', 'altura_garupa', 'profundidade_peito',
    'largura_peito', 'perimetro_toracico', 'largura_iliaca',
    'largura_isquiatica', 'comp_obliquo_corpo', 'comp_obliquo_traseiro',
    'perimetro_metacarpo', 'meia_circ_traseira', 'idade_anos'
]
TARGET = 'peso_kg'

NOMES_PT = {
    'altura_cernelha':       'Altura Cernelha (cm)',
    'altura_garupa':         'Altura Garupa (cm)',
    'profundidade_peito':    'Prof. Peito (cm)',
    'largura_peito':         'Larg. Peito (cm)',
    'perimetro_toracico':    'Perím. Torácico (cm)',
    'largura_iliaca':        'Larg. Ilíaca (cm)',
    'largura_isquiatica':    'Larg. Isquiática (cm)',
    'comp_obliquo_corpo':    'Comp. Oblíquo Corpo (cm)',
    'comp_obliquo_traseiro': 'Comp. Oblíquo Traseiro (cm)',
    'perimetro_metacarpo':   'Perím. Metacarpo (cm)',
    'meia_circ_traseira':    'Meia Circ. Traseira (cm)',
    'idade_anos':            'Idade (anos)',
    'area_corporal':   'Área Corporal (cm²)',
    'ratio_altura':    'Ratio Cernelha/Garupa',
    'volume_estimado': 'Volume Estimado (cm³)',
}

# Remove único valor nulo (perimetro_metacarpo)
df = df.dropna()

print(f'Dataset: {df.shape[0]} animais, {df.shape[1]} variáveis')
print(f'Features: {len(FEATURES)} | Target: {TARGET}')
print()
print(df[FEATURES + [TARGET]].describe().round(2))